# Multi-Model Operating Points & Athena-Matched Comparison

Loads trained models from a list of `results/` folders, selects the best trained model per kinematic region ($E_T \times |\eta|$), and evaluates $P_D$, $SP$, and $P_F$ for:
1. **Training Best SP** (validation-selected operating point)
2. **Calibrated Hard Cut** (matched to full-dataset Athena $P_D$)
3. **Linear Pileup-Corrected Cut $y_{\text{cut}}(\langle\mu\rangle)$** (matched to full-dataset Athena $P_D$)
4. **Athena Cut Baseline** (`loose`, `medium`, or `tight`)

Includes a `config_name` column in the tidy DataFrame so models can be filtered and compared directly.

In [16]:
from pathlib import Path
import sys
from pathlib import Path

PROJECT_ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

import pandas as pd
from src.core.Validation.RegionalOperatingPointsConsolidator import (
    MultiModelAthenaMatchedConsolidator,
)

print(f"Project root resolved: {PROJECT_ROOT}")

Project root resolved: /eos/home-i04/p/pmourafr/egamma_master


### 1. Configuration & Results Folders
Specify the list of results directories (`config_name<Name>_id<Timestamp>`). Configurations are auto-resolved from `config/NeuralRinger/`.

In [17]:
CONFIG_DIR = PROJECT_ROOT / "config/NeuralRinger"
RESULTS_FOLDERS = [
    PROJECT_ROOT / "results/config_nameModelV9_HighBatch_100Rings_id20261105222012",
    PROJECT_ROOT / "results/config_nameModelV9_HighBatch_50Rings_id20261109122259",
    PROJECT_ROOT / "results/config_nameModelV10_HighBatch_100Rings_id20261112210846",
    PROJECT_ROOT / "results/config_nameModelV10_HighBatch_50Rings_id20261115051346",
]
MODEL_LABELS = [
    "Model V9 (100 Rings)",
    "Model V9 (50 Rings)",
    "Model V10 (100 Rings)",
    "Model V10 (50 Rings)",
]
MANIFEST_PATH = PROJECT_ROOT / "results/split_manifest.json"

WORKING_POINT = "loose"
BINS_MU = 10
N_BOOTSTRAPS_PILEUP = 100
EFFICIENCIES_CSV = None
INCLUDE_TRAINING_BEST_SP = True

AS_PERCENTAGE = True
PRECISION = 2
DECIMAL_COMMA = False
MATH_DELIMITERS = True
BOLD_BEST_SP = True

OUTPUT_DIR = PROJECT_ROOT / "notebooks/Tables"
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
PARQUET_OUTPUT = OUTPUT_DIR / "multi_model_operating_points_tidy.parquet"
GRID_PARQUET_OUTPUT = OUTPUT_DIR / "multi_model_operating_points_grid.parquet"
LATEX_OUTPUT = OUTPUT_DIR / "multi_model_operating_points.tex"

for label, folder in zip(MODEL_LABELS, RESULTS_FOLDERS):
    print(f"{label}: {folder} (exists={folder.exists()})")

Model V9 (100 Rings): /eos/home-i04/p/pmourafr/egamma_master/results/config_nameModelV9_HighBatch_100Rings_id20261105222012 (exists=True)
Model V9 (50 Rings): /eos/home-i04/p/pmourafr/egamma_master/results/config_nameModelV9_HighBatch_50Rings_id20261109122259 (exists=True)
Model V10 (100 Rings): /eos/home-i04/p/pmourafr/egamma_master/results/config_nameModelV10_HighBatch_100Rings_id20261112210846 (exists=True)
Model V10 (50 Rings): /eos/home-i04/p/pmourafr/egamma_master/results/config_nameModelV10_HighBatch_50Rings_id20261115051346 (exists=True)


### 2. Evaluate All Models Across Regions
Instantiate `MultiModelAthenaMatchedConsolidator` from `RESULTS_FOLDERS` and run evaluation.

In [18]:
consolidator = MultiModelAthenaMatchedConsolidator.from_results_dirs(
    results_dirs=RESULTS_FOLDERS,
    config_dir=CONFIG_DIR,
    labels=MODEL_LABELS,
    manifest_path=MANIFEST_PATH if MANIFEST_PATH.exists() else None,
    working_point=WORKING_POINT,
    bins_mu=BINS_MU,
    n_bootstraps_pileup=N_BOOTSTRAPS_PILEUP,
    efficiencies_csv=EFFICIENCIES_CSV,
    include_training_best_sp=INCLUDE_TRAINING_BEST_SP,
)

path = Path(PARQUET_OUTPUT)
if not path.exists():
    records = consolidator.scan_and_evaluate()
    print(f"Total extracted records: {len(records)}")
    tidy_df = consolidator.build_tidy_dataframe()
else:
    tidy_df = pd.read_parquet(path)
    consolidator._records = tidy_df.to_dict("records")
    print(f"Loaded {len(tidy_df)} records from {path}")


Loaded 325 records from /eos/home-i04/p/pmourafr/egamma_master/notebooks/Tables/multi_model_operating_points_tidy.parquet


### 3. Tidy DataFrame & Model Selection by `config_name`
Inspect the tidy metrics table and filter by `config_name` to compare specific models.

In [19]:
print("Available config_name values:", tidy_df["config_name"].unique().tolist())
display(tidy_df[["iet", "ieta", "config_name", "method", "pd", "sp", "pf"]].head(15))

Available config_name values: ['ModelV9_HighBatch_100Rings', 'Athena', 'ModelV9_HighBatch_50Rings', 'ModelV10_HighBatch_100Rings', 'ModelV10_HighBatch_50Rings']


,iet,ieta,config_name,method,pd,sp,pf
0,0,0,ModelV9_HighBatch_100Rings,ModelV9_HighBatch_100Rings [Best SP],0.644144,0.662160,0.319575
1,0,0,ModelV9_HighBatch_100Rings,ModelV9_HighBatch_100Rings [Hard Cut],0.170915,0.486070,0.017689
2,0,0,ModelV9_HighBatch_100Rings,ModelV9_HighBatch_100Rings [$y_\mathrm{cut}(\l...,0.179910,0.491064,0.027123
3,0,0,Athena,Athena Loose Cut,0.202399,0.489955,0.086085
4,0,1,ModelV9_HighBatch_100Rings,ModelV9_HighBatch_100Rings [Best SP],0.686071,0.710613,0.264414
5,0,1,ModelV9_HighBatch_100Rings,ModelV9_HighBatch_100Rings [Hard Cut],0.219917,0.511498,0.065606
6,0,1,ModelV9_HighBatch_100Rings,ModelV9_HighBatch_100Rings [$y_\mathrm{cut}(\l...,0.217842,0.508405,0.069583
7,0,1,Athena,Athena Loose Cut,0.242739,0.490299,0.170974
8,0,2,ModelV9_HighBatch_100Rings,ModelV9_HighBatch_100Rings [Best SP],0.657143,0.730114,0.193069
9,0,2,ModelV9_HighBatch_100Rings,ModelV9_HighBatch_100Rings [Hard Cut],0.374408,0.615634,0.081683


### 4. 2D Kinematic Grid Comparison Table
Build and display the formatted $E_T \times |\eta|$ comparison grid across all loaded models.

In [20]:
grid_table = consolidator.build_grid_table(
    as_percentage=AS_PERCENTAGE,
    precision=PRECISION,
    decimal_comma=DECIMAL_COMMA,
    math_delimiters=MATH_DELIMITERS,
    bold_best_sp=BOLD_BEST_SP,
)
display(grid_table)

ET                                                                          $15 \le E_T < 20\,\text{GeV}$  \
Metric                                                                                            P_D [%]   
eta                      Method                                                                             
$0.00 \le |\eta| < 0.80$ Athena Loose Cut                                                $20.24 \pm 1.56$   
                         Model V9 (100 Rings) [Best SP]                                  $64.41 \pm 6.37$   
                         Model V9 (100 Rings) [Hard Cut]                                 $17.09 \pm 1.46$   
                         Model V9 (100 Rings) [$y_\mathrm{cut}(\langle\m...              $17.99 \pm 1.49$   
                         Model V9 (50 Rings) [Best SP]                          $\mathbf{64.17 \pm 3.43}$   
...                                                                                                   ...   
$2.37 \le |\eta| < 2.50$ Model V10 (100 Rings) [Hard Cut]                                $13.21 \pm 4.65$   
                         Model V10 (100 Rings) [$y_\mathrm{cut}(\langle\...              $15.09 \pm 4.92$   
                         Model V10 (50 Rings) [Best SP]                                 $81.13 \pm 10.58$   
                         Model V10 (50 Rings) [Hard Cut]                                 $13.21 \pm 4.65$   
                         Model V10 (50 Rings) [$y_\mathrm{cut}(\langle\m...              $15.09 \pm 4.92$   

ET                                                                                                      \
Metric                                                                                          SP [%]   
eta                      Method                                                                          
$0.00 \le |\eta| < 0.80$ Athena Loose Cut                                             $49.00 \pm 1.33$   
                         Model V9 (100 Rings) [Best SP]                               $66.22 \pm 1.12$   
                         Model V9 (100 Rings) [Hard Cut]                              $48.61 \pm 1.35$   
                         Model V9 (100 Rings) [$y_\mathrm{cut}(\langle\m...           $49.11 \pm 1.35$   
                         Model V9 (50 Rings) [Best SP]                       $\mathbf{66.48 \pm 1.47}$   
...                                                                                                ...   
$2.37 \le |\eta| < 2.50$ Model V10 (100 Rings) [Hard Cut]                             $45.36 \pm 4.92$   
                         Model V10 (100 Rings) [$y_\mathrm{cut}(\langle\...           $47.28 \pm 4.86$   
                         Model V10 (50 Rings) [Best SP]                               $75.46 \pm 3.48$   
                         Model V10 (50 Rings) [Hard Cut]                              $45.36 \pm 4.92$   
                         Model V10 (50 Rings) [$y_\mathrm{cut}(\langle\m...           $47.28 \pm 4.86$   

ET                                                                                                      \
Metric                                                                                         P_F [%]   
eta                      Method                                                                          
$0.00 \le |\eta| < 0.80$ Athena Loose Cut                                              $8.61 \pm 0.96$   
                         Model V9 (100 Rings) [Best SP]                               $31.96 \pm 6.26$   
                         Model V9 (100 Rings) [Hard Cut]                               $1.77 \pm 0.45$   
                         Model V9 (100 Rings) [$y_\mathrm{cut}(\langle\m...            $2.71 \pm 0.56$   
                         Model V9 (50 Rings) [Best SP]                       $\mathbf{31.17 \pm 4.43}$   
...                                                                                                ...   
$2.37 \le |\eta| < 2.50$ Model V10

In [39]:
grid_table

ET                                                                          $15 \le E_T < 20\,\text{GeV}$  \
Metric                                                                                            P_D [%]   
eta                      Method                                                                             
$0.00 \le |\eta| < 0.80$ Athena Loose Cut                                                $20.24 \pm 1.56$   
                         Model V9 (100 Rings) [Best SP]                                  $64.41 \pm 6.37$   
                         Model V9 (100 Rings) [Hard Cut]                                 $17.09 \pm 1.46$   
                         Model V9 (100 Rings) [$y_\mathrm{cut}(\langle\m...              $17.99 \pm 1.49$   
                         Model V9 (50 Rings) [Best SP]                          $\mathbf{64.17 \pm 3.43}$   
...                                                                                                   ...   
$2.37 \le |\eta| < 2.50$ Model V10 (100 Rings) [Hard Cut]                                $13.21 \pm 4.65$   
                         Model V10 (100 Rings) [$y_\mathrm{cut}(\langle\...              $15.09 \pm 4.92$   
                         Model V10 (50 Rings) [Best SP]                                 $81.13 \pm 10.58$   
                         Model V10 (50 Rings) [Hard Cut]                                 $13.21 \pm 4.65$   
                         Model V10 (50 Rings) [$y_\mathrm{cut}(\langle\m...              $15.09 \pm 4.92$   

ET                                                                                                      \
Metric                                                                                          SP [%]   
eta                      Method                                                                          
$0.00 \le |\eta| < 0.80$ Athena Loose Cut                                             $49.00 \pm 1.33$   
                         Model V9 (100 Rings) [Best SP]                               $66.22 \pm 1.12$   
                         Model V9 (100 Rings) [Hard Cut]                              $48.61 \pm 1.35$   
                         Model V9 (100 Rings) [$y_\mathrm{cut}(\langle\m...           $49.11 \pm 1.35$   
                         Model V9 (50 Rings) [Best SP]                       $\mathbf{66.48 \pm 1.47}$   
...                                                                                                ...   
$2.37 \le |\eta| < 2.50$ Model V10 (100 Rings) [Hard Cut]                             $45.36 \pm 4.92$   
                         Model V10 (100 Rings) [$y_\mathrm{cut}(\langle\...           $47.28 \pm 4.86$   
                         Model V10 (50 Rings) [Best SP]                               $75.46 \pm 3.48$   
                         Model V10 (50 Rings) [Hard Cut]                              $45.36 \pm 4.92$   
                         Model V10 (50 Rings) [$y_\mathrm{cut}(\langle\m...           $47.28 \pm 4.86$   

ET                                                                                                      \
Metric                                                                                         P_F [%]   
eta                      Method                                                                          
$0.00 \le |\eta| < 0.80$ Athena Loose Cut                                              $8.61 \pm 0.96$   
                         Model V9 (100 Rings) [Best SP]                               $31.96 \pm 6.26$   
                         Model V9 (100 Rings) [Hard Cut]                               $1.77 \pm 0.45$   
                         Model V9 (100 Rings) [$y_\mathrm{cut}(\langle\m...            $2.71 \pm 0.56$   
                         Model V9 (50 Rings) [Best SP]                       $\mathbf{31.17 \pm 4.43}$   
...                                                                                                ...   
$2.37 \le |\eta| < 2.50$ Model V10

In [47]:
search_terms = ["Athena Loose Cut"] + ["100"]

pattern = '|'.join(search_terms)
mask = grid_table.index.get_level_values('Method').str.contains(pattern, na=False)
df_RingsModelsFiltered = grid_table[mask]

### 5. Export Parquet & LaTeX Tables
Save the tidy DataFrame (with `config_name`), the 2D grid table, and the publication-ready LaTeX snippet.

In [49]:
# saved_tidy = consolidator.export_parquet(PARQUET_OUTPUT, df=tidy_df)
# saved_grid = consolidator.export_parquet(GRID_PARQUET_OUTPUT, df=grid_table)

print(f"Using mask: {search_terms}")
latex_code = consolidator.to_latex_snippet(
    grid_table=df_RingsModelsFiltered,
    caption=(
        rf"Comparison of Athena {WORKING_POINT.capitalize()} Cut, Training Best SP, "
        rf"Calibrated Hard Cut, and Linear Pileup-Corrected Cut $y_\mathrm{{cut}}(\langle\mu\rangle)$ "
        rf"across $E_T$ and $|\eta|$ regions."
    ),
    label="tab:multi_model_operating_points_comparison",
    resize_box=True,
    as_percentage=AS_PERCENTAGE,
    precision=PRECISION,
    decimal_comma=DECIMAL_COMMA,
)
saved_latex = consolidator.export_latex(LATEX_OUTPUT, latex_code=latex_code)

# print(f"Tidy Parquet saved to: {saved_tidy}")
# print(f"Grid Parquet saved to: {saved_grid}")
print(f"LaTeX saved to:        {saved_latex}")

Using mask: ['Athena Loose Cut', '100']
LaTeX saved to:        /eos/home-i04/p/pmourafr/egamma_master/notebooks/Tables/multi_model_operating_points.tex


### 6. Mean Performance Summary Across Regions ($P_D$, $SP$, $P_F$)
Computes the mean (and standard deviation across kinematic regions) of $P_D$, $SP$, and $P_F$ for each `config_name` and operating point method.

In [50]:
scale = 100.0 if AS_PERCENTAGE else 1.0
unit = "[%]" if AS_PERCENTAGE else ""

summary_df = (
    tidy_df.groupby(["config_name", "method"], sort=False)[["pd", "sp", "pf"]]
    .agg(["mean", "std"])
    .mul(scale)
    .round(PRECISION)
)
summary_df.columns = [
    f"{metric.upper()} {stat} {unit}".strip()
    for metric, stat in summary_df.columns
]
summary_df = summary_df.reset_index()
display(summary_df.sort_values(by="PF mean [%]"))

summary_parquet = consolidator.export_parquet(
    OUTPUT_DIR / "multi_model_mean_summary.parquet",
    df=summary_df,
)
print(f"Mean summary Parquet saved to: {summary_parquet}")

,config_name,method,PD mean [%],PD std [%],SP mean [%],SP std [%],PF mean [%],PF std [%]
2,ModelV9_HighBatch_100Rings,ModelV9_HighBatch_100Rings [$y_\mathrm{cut}(\l...,64.02,23.78,76.02,13.61,8.22,6.79
1,ModelV9_HighBatch_100Rings,ModelV9_HighBatch_100Rings [Hard Cut],63.90,23.82,75.92,13.68,8.25,6.81
5,ModelV9_HighBatch_50Rings,ModelV9_HighBatch_50Rings [Hard Cut],63.90,23.82,75.67,13.60,8.78,7.07
6,ModelV9_HighBatch_50Rings,ModelV9_HighBatch_50Rings [$y_\mathrm{cut}(\la...,64.15,23.94,75.78,13.67,8.85,7.07
8,ModelV10_HighBatch_100Rings,ModelV10_HighBatch_100Rings [Hard Cut],63.90,23.82,75.57,13.56,8.99,7.21
9,ModelV10_HighBatch_100Rings,ModelV10_HighBatch_100Rings [$y_\mathrm{cut}(\...,64.17,23.73,75.74,13.46,9.03,7.14
11,ModelV10_HighBatch_50Rings,ModelV10_HighBatch_50Rings [Hard Cut],63.90,23.82,75.32,13.48,9.51,7.62
12,ModelV10_HighBatch_50Rings,ModelV10_HighBatch_50Rings [$y_\mathrm{cut}(\l...,64.27,23.60,75.54,13.26,9.56,7.64
0,ModelV9_HighBatch_100Rings,ModelV9_HighBatch_100Rings [Best SP],81.36,7.34,83.20,6.46,14.90,6.04
4,ModelV9_HighBatch_50Rings,ModelV9_HighBatch_50Rings [Best SP],81.54,6.51,82.79,6.41,15.91,7.02


Mean summary Parquet saved to: /eos/home-i04/p/pmourafr/egamma_master/notebooks/Tables/multi_model_mean_summary.parquet
